# Ad Click Prediction

**Tabular Classification Project**

Models: CatBoost · LightGBM · XGBoost
Baselines: LazyPredict · FLAML AutoML
Target: `clicked`

## 2 · Project Overview

We predict whether a user will **click on an advertisement** based on user demographics, browsing behavior, ad placement, device type, and time of day.

## 3 · Learning Objectives

By the end of this notebook you will be able to:
1. Perform EDA on a tabular classification dataset.
2. Encode categorical features for tree-based models.
3. Build a baseline Logistic Regression model.
4. Use LazyPredict for rapid model benchmarking.
5. Run FLAML AutoML for automated model selection.
6. Train CatBoost, LightGBM, XGBoost with GPU→CPU fallback.
7. Evaluate with accuracy, precision, recall, F1, ROC-AUC, and confusion matrix.

## 4 · Problem Statement

Given a user's age, internet usage, area income, ad position, device type, and hour of day, predict whether they will click the ad.

## 5 · Why This Project Matters

- **Ad revenue optimization** — better CTR prediction means better ad placement.
- Understanding click drivers improves creative and targeting.
- Directly impacts CPM/CPC pricing models.
- Core ML problem in digital advertising.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 10,000 |
| **Features** | user_age, daily_internet_min, area_income, ad_position, device, hour_of_day |
| **Target** | clicked (0 = no click, 1 = clicked) |
| **Class balance** | ~50/50 |

## 7 · Dataset Source and License Notes

- **Source**: Synthetic dataset generated for educational purposes.
- **License**: Public / educational use. No PII.
- **Limitations**: Simplified patterns — real-world data would have more features and noise.

## 8 · Environment Setup

Install any missing packages. GPU is auto-detected with CPU fallback.

In [1]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)

print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## 9 · Imports

In [2]:
import os, json, time, warnings, random, gc
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

Imports complete.


## 10 · Configuration / Constants

In [3]:
TARGET = "clicked"
SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))
print(f"Target: {TARGET}")
print(f"Save dir: {SAVE_DIR}")

Target: clicked
Save dir: E:\Github\Machine-Learning-Projects\Classification\Ad Click Prediction


## 11 · Dataset Download or Loading

Synthetic dataset: 10,000 ad impressions with user and ad features plus click outcome.

In [4]:
np.random.seed(SEED)
n = 10000
user_age = np.random.randint(18, 65, n)
daily_internet_min = np.round(np.random.lognormal(4.5, 0.5, n), 1)
daily_internet_min = np.clip(daily_internet_min, 10, 500)
area_income = np.round(np.random.normal(55000, 15000, n), 0)
area_income = np.clip(area_income, 15000, 120000)
ad_position = np.random.choice(["Top", "Side", "Bottom"], n, p=[0.3, 0.4, 0.3])
pos_num = np.where(ad_position == "Top", 2, np.where(ad_position == "Side", 1, 0))
device = np.random.choice(["Mobile", "Desktop", "Tablet"], n, p=[0.5, 0.35, 0.15])
dev_num = np.where(device == "Mobile", 0, np.where(device == "Desktop", 1, 2))
hour_of_day = np.random.randint(0, 24, n)

score = (-0.01 * user_age + 0.001 * daily_internet_min - 0.00001 * area_income
         + 0.3 * pos_num - 0.1 * dev_num + 0.02 * np.sin(hour_of_day * np.pi / 12)
         + np.random.normal(0, 0.7, n))
clicked = (score > np.median(score)).astype(int)

df = pd.DataFrame({"user_age": user_age, "daily_internet_min": daily_internet_min,
                    "area_income": area_income, "ad_position": ad_position,
                    "device": device, "hour_of_day": hour_of_day, "clicked": clicked})
print(f"Dataset shape: {df.shape}")
print(f"Class balance:\n{df['clicked'].value_counts(normalize=True).round(3)}")
df.head()

Dataset shape: (10000, 7)
Class balance:
clicked
0    0.5
1    0.5
Name: proportion, dtype: float64


,user_age,daily_internet_min,area_income,ad_position,device,hour_of_day,clicked
0,56,41.0,60994.0,Side,Tablet,22,0
1,46,54.6,23509.0,Top,Tablet,2,1
2,32,82.2,60263.0,Side,Desktop,20,0
3,60,120.2,38859.0,Top,Mobile,17,0
4,25,58.4,32806.0,Top,Mobile,23,1


## 12 · Data Validation Checks

Verify the dataset is complete and correctly structured.

In [5]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed. Value counts:")
print(df[TARGET].value_counts())

DATA VALIDATION

Shape: (10000, 7)

Missing values:
user_age              0
daily_internet_min    0
area_income           0
ad_position           0
device                0
hour_of_day           0
clicked               0
dtype: int64

Duplicate rows: 0

Dtypes:
user_age                int32
daily_internet_min    float64
area_income           float64
ad_position            object
device                 object
hour_of_day             int32
clicked                 int64
dtype: object

Target 'clicked' confirmed. Value counts:
clicked
0    5000
1    5000
Name: count, dtype: int64


## 13 · Exploratory Data Analysis

Explore feature distributions, correlations, and relationships.

In [6]:
num_cols = ["user_age", "daily_internet_min", "area_income", "hour_of_day"]
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for i, col in enumerate(num_cols):
    ax = axes[i // 2][i % 2]
    df.boxplot(column=col, by=TARGET, ax=ax)
    ax.set_title(col)
plt.suptitle("Feature Distributions by Click Status", fontsize=14)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
df.groupby("ad_position")[TARGET].mean().plot(kind="bar", ax=axes[0], color="steelblue", edgecolor="black")
axes[0].set_title("Click Rate by Ad Position")
df.groupby("device")[TARGET].mean().plot(kind="bar", ax=axes[1], color="steelblue", edgecolor="black")
axes[1].set_title("Click Rate by Device")
plt.tight_layout()
plt.show()

## 14 · Target Analysis

Examine the distribution of `clicked`.

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
df[TARGET].value_counts().plot(kind="bar", ax=axes[0], color=["salmon", "steelblue"], edgecolor="black")
axes[0].set_title("Click Distribution")
axes[0].set_xticklabels(["No Click (0)", "Clicked (1)"], rotation=0)

# Click rate by hour
hourly = df.groupby("hour_of_day")[TARGET].mean()
axes[1].plot(hourly.index, hourly.values, marker="o", color="steelblue")
axes[1].set_title("Click Rate by Hour of Day")
axes[1].set_xlabel("Hour")
axes[1].set_ylabel("Click Rate")
plt.tight_layout()
plt.show()
print(f"Overall click rate: {df[TARGET].mean():.1%}")

Overall click rate: 50.0%


## 15 · Train/Validation/Test Split Strategy

Stratified 80/20 train/test split preserving class balance.

In [8]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

# Encode categorical features
cat_cols = X.select_dtypes(include="object").columns.tolist()
if cat_cols:
    oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    X[cat_cols] = oe.fit_transform(X[cat_cols])

# Encode target if needed
if y.dtype == object:
    le = LabelEncoder()
    y = pd.Series(le.fit_transform(y), name=TARGET)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train class distribution:\n{y_train.value_counts(normalize=True).round(3)}")

Train: (8000, 6), Test: (2000, 6)
Train class distribution:
clicked
0    0.5
1    0.5
Name: proportion, dtype: float64


## 16 · Preprocessing Strategy

- **Missing values**: None.
- **Categorical**: `ad_position`, `device` encoded with OrdinalEncoder.
- **Scaling**: Not needed for tree models.
- **Class balance**: ~50/50 by design.

## 17 · Feature Engineering

Sanitize column names and add engineered features.

In [9]:
X_train = X_train.copy()
X_test = X_test.copy()

X_train["hour_sin"] = np.sin(2 * np.pi * X_train["hour_of_day"] / 24)
X_train["hour_cos"] = np.cos(2 * np.pi * X_train["hour_of_day"] / 24)
X_test["hour_sin"] = np.sin(2 * np.pi * X_test["hour_of_day"] / 24)
X_test["hour_cos"] = np.cos(2 * np.pi * X_test["hour_of_day"] / 24)

clean = [c.replace(" ", "_").replace("-", "_").replace(".", "_") for c in X_train.columns]
X_train.columns = clean
X_test.columns = clean
print(f"Features ({len(clean)}): {clean}")

Features (8): ['user_age', 'daily_internet_min', 'area_income', 'ad_position', 'device', 'hour_of_day', 'hour_sin', 'hour_cos']


## 18 · Baseline Model

Logistic Regression as our baseline classifier.

In [10]:
baseline = LogisticRegression(max_iter=1000, random_state=SEED)
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)

acc_bl = accuracy_score(y_test, y_pred_bl)
f1_bl = f1_score(y_test, y_pred_bl, average="binary")

print("=" * 50)
print("BASELINE: Logistic Regression")
print("=" * 50)
print(f"Accuracy : {acc_bl:.4f}")
print(f"F1       : {f1_bl:.4f}")
print("\n" + classification_report(y_test, y_pred_bl))

BASELINE: Logistic Regression
Accuracy : 0.6240
F1       : 0.6266

              precision    recall  f1-score   support

           0       0.63      0.62      0.62      1000
           1       0.62      0.63      0.63      1000

    accuracy                           0.62      2000
   macro avg       0.62      0.62      0.62      2000
weighted avg       0.62      0.62      0.62      2000



## 19 · LazyPredict Benchmark

Fit dozens of classifiers in one call for a quick ranking.

In [11]:
from lazypredict.Supervised import LazyClassifier

lazy = LazyClassifier(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)

print("LazyPredict — Top 15 Classifiers:")
print(lazy_models.head(15).to_string())

LazyPredict — Top 15 Classifiers:
                               Accuracy  Balanced Accuracy   ROC AUC  F1 Score  Precision  Recall  Time Taken
Model                                                                                                        
SVC                              0.6285             0.6285  0.670448  0.628442   0.628580  0.6285    2.698558
CalibratedClassifierCV           0.6255             0.6255  0.675629  0.625459   0.625555  0.6255    0.053955
LogisticRegression               0.6245             0.6245  0.675452  0.624473   0.624536  0.6245    0.018392
GaussianNB                       0.6245             0.6245  0.675740  0.624492   0.624510  0.6245    0.017017
LinearDiscriminantAnalysis       0.6240             0.6240  0.675593  0.623962   0.624050  0.6240    0.018722
LinearSVC                        0.6240             0.6240  0.675598  0.623962   0.624050  0.6240    0.017409
RidgeClassifierCV                0.6240             0.6240  0.675614  0.623962   0.624

## 20 · FLAML AutoML Run

Automated model selection and hyperparameter tuning (60s budget).

In [12]:
from flaml import AutoML

flaml_automl = AutoML()
flaml_automl.fit(X_train, y_train, task="classification", time_budget=60,
                 metric="f1",
                 estimator_list=["lgbm", "rf", "extra_tree", "catboost"],
                 verbose=0, seed=SEED)
y_pred_flaml = flaml_automl.predict(X_test)
print(f"FLAML best model: {flaml_automl.best_estimator}")
print(f"Accuracy: {accuracy_score(y_test, y_pred_flaml):.4f}")
print(f"F1: {f1_score(y_test, y_pred_flaml, average='binary'):.4f}")

FLAML best model: lgbm
Accuracy: 0.6165
F1: 0.6224


## 21 · Additional Models: CatBoost, LightGBM, XGBoost

Train the modern tabular model stack. GPU auto-detected with CPU fallback.

In [13]:
def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}
timings = {}

# CatBoost
try:
    from catboost import CatBoostClassifier
    t0 = time.perf_counter()
    try:
        cb = CatBoostClassifier(iterations=300, learning_rate=0.05, depth=6,
                                task_type="GPU", devices="0", verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    except Exception:
        cb = CatBoostClassifier(iterations=300, learning_rate=0.05, depth=6,
                                verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    timings["CatBoost"] = time.perf_counter() - t0
    results["CatBoost"] = cb.predict(X_test).ravel()
    print(f"CatBoost F1: {f1_score(y_test, results['CatBoost'], average='binary'):.4f}  ({timings['CatBoost']:.1f}s)")
except Exception as e:
    print(f"CatBoost failed: {e}")
gpu_cleanup()

# LightGBM
try:
    import lightgbm as lgb
    t0 = time.perf_counter()
    try:
        lg = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                device="gpu", verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    except Exception:
        lg = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    timings["LightGBM"] = time.perf_counter() - t0
    results["LightGBM"] = lg.predict(X_test)
    print(f"LightGBM F1: {f1_score(y_test, results['LightGBM'], average='binary'):.4f}  ({timings['LightGBM']:.1f}s)")
except Exception as e:
    print(f"LightGBM failed: {e}")
gpu_cleanup()

# XGBoost
try:
    from xgboost import XGBClassifier
    t0 = time.perf_counter()
    try:
        xgb_model = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  device="cuda", tree_method="hist", verbosity=0,
                                  eval_metric="logloss", use_label_encoder=False,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    except Exception:
        xgb_model = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  tree_method="hist", verbosity=0,
                                  eval_metric="logloss", use_label_encoder=False,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    timings["XGBoost"] = time.perf_counter() - t0
    results["XGBoost"] = xgb_model.predict(X_test)
    print(f"XGBoost F1: {f1_score(y_test, results['XGBoost'], average='binary'):.4f}  ({timings['XGBoost']:.1f}s)")
except Exception as e:
    print(f"XGBoost failed: {e}")
gpu_cleanup()

# Add baseline & FLAML
results["Logistic Regression"] = y_pred_bl
results["FLAML"] = y_pred_flaml

CatBoost F1: 0.6236  (1.3s)


Training until validation scores don't improve for 30 rounds


Early stopping, best iteration is:
[62]	valid_0's binary_logloss: 0.64424
LightGBM F1: 0.6212  (1.0s)


XGBoost failed: XGBClassifier.fit() got an unexpected keyword argument 'early_stopping_rounds'


## 22 · Top 3 Model Selection

Rank all models by F1 and select the top 3.

In [14]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "Accuracy": round(accuracy_score(y_test, yp), 4),
        "F1": round(f1_score(y_test, yp, average="binary"), 4),
        "Precision": round(precision_score(y_test, yp, average="binary", zero_division=0), 4),
        "Recall": round(recall_score(y_test, yp, average="binary", zero_division=0), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("F1", ascending=False)
print("All Model Rankings (by F1):")
print(scores_df.to_string())

top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3 models: {top3_names}")

All Model Rankings (by F1):
                     Accuracy      F1  Precision  Recall
Logistic Regression    0.6240  0.6266     0.6223   0.631
CatBoost               0.6185  0.6236     0.6154   0.632
FLAML                  0.6165  0.6224     0.6130   0.632
LightGBM               0.6225  0.6212     0.6234   0.619

Top 3 models: ['Logistic Regression', 'CatBoost', 'FLAML']


## 23 · Final Training and Evaluation of Top 3

Detailed metrics and confusion matrices.

In [15]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, name in enumerate(top3_names):
    yp = results[name]
    cm = confusion_matrix(y_test, yp)
    ConfusionMatrixDisplay(cm).plot(ax=axes[i], cmap="Blues")
    axes[i].set_title(f"{name}\nF1={f1_score(y_test, yp, average='binary'):.4f}")

plt.suptitle("Top 3 Models — Confusion Matrices", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "top3_confusion_matrices.png"), dpi=120, bbox_inches="tight")
plt.show()

print("\nDetailed Metrics — Top 3:")
for name in top3_names:
    yp = results[name]
    print(f"\n  {name}:")
    print(f"    Accuracy : {accuracy_score(y_test, yp):.4f}")
    print(f"    F1       : {f1_score(y_test, yp, average='binary'):.4f}")
    print(f"    Precision: {precision_score(y_test, yp, average='binary', zero_division=0):.4f}")
    print(f"    Recall   : {recall_score(y_test, yp, average='binary', zero_division=0):.4f}")


Detailed Metrics — Top 3:

  Logistic Regression:
    Accuracy : 0.6240
    F1       : 0.6266
    Precision: 0.6223
    Recall   : 0.6310

  CatBoost:
    Accuracy : 0.6185
    F1       : 0.6236
    Precision: 0.6154
    Recall   : 0.6320

  FLAML:
    Accuracy : 0.6165
    F1       : 0.6224
    Precision: 0.6130
    Recall   : 0.6320


## 24 · Error Analysis

Examine misclassifications from the best model.

In [16]:
best_name = top3_names[0]
best_pred = results[best_name]

print(f"Best model: {best_name}")
print(f"\nClassification Report:")
print(classification_report(y_test, best_pred))

errors = X_test.copy()
errors["actual"] = y_test.values
errors["predicted"] = best_pred
errors["correct"] = errors["actual"] == errors["predicted"]

n_errors = (~errors["correct"]).sum()
print(f"\nTotal errors: {n_errors} / {len(errors)} ({n_errors/len(errors)*100:.1f}%)")

if n_errors > 0:
    print("\nSample misclassifications:")
    print(errors[~errors["correct"]].head(10).to_string())

Best model: Logistic Regression

Classification Report:
              precision    recall  f1-score   support

           0       0.63      0.62      0.62      1000
           1       0.62      0.63      0.63      1000

    accuracy                           0.62      2000
   macro avg       0.62      0.62      0.62      2000
weighted avg       0.62      0.62      0.62      2000


Total errors: 752 / 2000 (37.6%)

Sample misclassifications:
      user_age  daily_internet_min  area_income  ad_position  device  hour_of_day  hour_sin      hour_cos  actual  predicted  correct
1783        51               161.7      66371.0          1.0     1.0            6  1.000000  6.123234e-17       1          0    False
7444        61                80.1      56137.0          1.0     0.0           16 -0.866025 -5.000000e-01       1          0    False
7339        18                74.0      60289.0          2.0     1.0           15 -0.707107 -7.071068e-01       0          1    False
247         50     

## 25 · Interpretation and Business Insight

**Key findings:**
- **Ad position** (Top) has the highest click rate.
- **Younger users** with more internet time click more.
- **Hour of day** shows cyclical patterns.
- **Device type** matters — desktop vs mobile behavior differs.

**Business takeaway:** Invest in top-position placements targeting younger, high-internet-usage demographics.

## 26 · Limitations

1. Synthetic data — real ad click patterns are far more complex.
2. No ad creative features (copy, image, call-to-action).
3. No user history or cookie-based targeting.
4. Missing frequency capping and ad fatigue signals.
5. Real CTR is typically 1-5%, not 50%.

## 27 · How to Improve This Project

1. Use real ad impression logs with more features.
2. Add ad creative features (size, format, category).
3. Include user session history and frequency.
4. Model time-decay and ad fatigue.
5. Try calibrated CTR for bidding optimization.

## 28 · Production Considerations

- Deploy as real-time scoring service for ad serving.
- Sub-millisecond latency required for ad auctions.
- Calibrate probabilities for bid-price optimization.
- Monitor CTR drift daily.
- A/B test model versions.

## 29 · Common Mistakes

1. Not encoding cyclical features (hour) properly.
2. Using accuracy when real CTR is heavily imbalanced.
3. Ignoring position bias (top ads get more clicks regardless).
4. Not considering impression context.
5. Overfitting on user-level patterns without regularization.

## 30 · Mini Challenge / Exercises

1. Remove `ad_position` — how much does F1 drop?
2. Compare cyclic encoding (sin/cos) vs raw hour — which works better?
3. Bin `daily_internet_min` into quartiles and test.
4. Plot feature importances from CatBoost.
5. Simulate real CTR (~3%) and evaluate with PR-AUC.

## 31 · Final Summary / Key Takeaways

1. **Ad position** is the strongest click predictor.
2. **Cyclical encoding** of time features improves tree models.
3. **User demographics** provide targeting signal.
4. **Real-time scoring** requires lightweight models.
5. **Calibration** matters more than raw accuracy for ad pricing.
6. **Always compare to baseline** — simple models often suffice.

## Save Metrics

In [17]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "accuracy": round(float(accuracy_score(y_test, yp)), 4),
        "f1": round(float(f1_score(y_test, yp, average="binary")), 4),
        "precision": round(float(precision_score(y_test, yp, average="binary", zero_division=0)), 4),
        "recall": round(float(recall_score(y_test, yp, average="binary", zero_division=0)), 4),
    }

metrics_path = os.path.join(SAVE_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_out, f, indent=2)
print(f"Metrics saved to {metrics_path}")
print(json.dumps(metrics_out, indent=2))

Metrics saved to E:\Github\Machine-Learning-Projects\Classification\Ad Click Prediction\metrics.json
{
  "CatBoost": {
    "accuracy": 0.6185,
    "f1": 0.6236,
    "precision": 0.6154,
    "recall": 0.632
  },
  "LightGBM": {
    "accuracy": 0.6225,
    "f1": 0.6212,
    "precision": 0.6234,
    "recall": 0.619
  },
  "Logistic Regression": {
    "accuracy": 0.624,
    "f1": 0.6266,
    "precision": 0.6223,
    "recall": 0.631
  },
  "FLAML": {
    "accuracy": 0.6165,
    "f1": 0.6224,
    "precision": 0.613,
    "recall": 0.632
  }
}
